# Python API Example - Gas Transit Costs

This guide is designed to provide an example of how to access the Spark API:
- The path to your client credentials is the only input needed to run this script (just before Section 2)
- This script has been designed to display the raw outputs of requests from the API, and then shows you how to format those outputs to enable easy reading and analysis
- This script can be copied and pasted by customers for quick use of the API

__N.B. This guide is just for Gas Transit data. If you're looking for other API data products (such as contract prices, Freight routes or Netbacks), please refer to their according code example files.__ 

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint: https://www.sparkcommodities.com/api/request/access.html

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/introduction.html


In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)


# 2. Reference Data

Here we fetch the relevant reference data for the endpoint - this is a list of all available transit route options available on the endpoint. Countries for each hub are also included for easy reference.

In [ ]:
from io import StringIO

## Defining the function

def fetch_ref_data(access_token, format='json'):

    uri="v1.0/gas/transit/reference-data/"
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )
    
    if format == 'json':
        my_dict = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            my_dict = content
            print('No Data to Show')
        else:
            my_dict = content.decode('utf-8')
            my_dict = pd.read_csv(StringIO(my_dict))  # automatically converting into a Pandas DataFrame when choosing CSV format
    
    return my_dict

ref_df = fetch_ref_data(access_token, format='csv')
ref_df

# 3. Calling Transit Costs data

Here we define a function to fetch the Gas Transit costs dataset. The available parameters can be found on our API docs:

https://www.sparkcommodities.com/api/gas/transit.html

__N.B.:__ This endpoint provides the option to return a JSON or CSV formatted response. Metadata is only available in the JSON format.

In [ ]:

def fetch_transit_prices(access_token, unit, origin=None, destination=None, start=None, end=None, limit=None, offset=None, 
                        booking_type=None, product_type=None, format='json'):
    
    query_params = "?unit={}".format(unit)

    if origin is not None:
        query_params += "&origin={}".format(origin)

    if destination is not None:
        query_params += "&destination={}".format(destination)

    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)
    
    if limit is not None:
        query_params += "&limit={}".format(limit)
    if offset is not None:
        query_params += "&offset={}".format(offset)

    if booking_type is not None:    
        query_params += "&type={}".format(booking_type)
    
    if product_type is not None:
        query_params += "&product={}".format(product_type)

    content = do_api_get_query(
        uri="/v1.0/gas/transit/costs/{}".format(query_params),
        access_token=access_token, format=format
    )

    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format

    return data

#### JSON Response

__N.B.__ The JSON response has a limited historical API call range of approximately 10 days (with no filter params applied) due to API dataset size limitations. Please use the CSV format to call up to 1 year's worth of historical data

In [ ]:
# Calling historical data - JSON response
json_raw = fetch_transit_prices(access_token, unit='eur-per-mwh', start='2026-05-17', end='2026-05-27', format='json')
json_raw

In [ ]:
# checking metaData
json_raw['metaData']

#### CSV Response

In [ ]:
# Calling historical data - CSV response
csv_df = fetch_transit_prices(access_token, unit='eur-per-mwh', start='2026-01-01', end='2026-05-27', format='csv')
csv_df

In [ ]:
# listing the column names of the DataFrame
csv_df.columns

## N.B. Historical Data Limits

Currently, a maximum of 1 year's worth of historical data can be called at one time due to the size of the data file. 

If more data points are required, the below code can be used: the function calls the historical data 1 year at a time and combines the data into one Pandas DataFrame

In [ ]:
import datetime

def loop_historical_data(access_token, unit, start, end, origin=None, destination=None, booking_type=None, product_type=None):
    
    hist_diff = (datetime.datetime.strptime(end, '%Y-%m-%d') - datetime.datetime.strptime(start, '%Y-%m-%d')).days
    t = 0

    w = 365
    print(hist_diff)
    while t < hist_diff:
        print(t)
        # initialising dataframe
        if t == 0 and hist_diff>w:
            diff_end = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
            hist_df = fetch_transit_prices(access_token, unit=unit, start=start, end=diff_end, origin=origin, destination=destination,
                        booking_type=booking_type, product_type=product_type, format='csv')
        
        elif t==0 and hist_diff<=w:
            hist_df = fetch_transit_prices(access_token, unit=unit, start=start, end=end, origin=origin, destination=destination,
                        booking_type=booking_type, product_type=product_type, format='csv')

        # appending additional historical data
        else:
            if t < hist_diff-w:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
                historical_addition = fetch_transit_prices(access_token, unit=unit, start=diff_start, end=diff_end, origin=origin, destination=destination,
                        booking_type=booking_type, product_type=product_type, format='csv')
                try:
                    hist_df = pd.concat([hist_df,historical_addition])
                #exception if hist_df is empty
                except:
                    hist_df = historical_addition.copy()

            else:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=hist_diff-t), '%Y-%m-%d')
                historical_addition = fetch_transit_prices(access_token, unit=unit, start=diff_start, end=diff_end, origin=origin, destination=destination,
                        booking_type=booking_type, product_type=product_type, format='csv')
                hist_df = pd.concat([hist_df, historical_addition])
        
        #looping by year
        t += w

    for c in list(hist_df.columns)[13:]:
        hist_df[c] = pd.to_numeric(hist_df[c])
    hist_df['ReleaseDate'] = pd.to_datetime(hist_df['ReleaseDate'])

    return hist_df

In [ ]:
"""
hist_df = loop_historical_data(access_token, unit='eur-per-mwh', start='2026-01-01', end='2026-05-30', 
                                origin=None, destination=None, booking_type=None, product_type=None)

hist_df
"""